# 03 — Simulation on Mock Data (DataLoaderMock)

Run the full simulation pipeline using `DataLoaderMock` → `DataLoaderGraph` → `build_model()` → `Environment.run()`.

This demonstrates a realistic-scale simulation (10 stations, 2 depots, 168 hourly timestamps)
instead of the minimal hand-crafted fixtures used in `02_environment_skeleton.ipynb`.

In [ ]:
from gbp.loaders import DataLoaderMock, DataLoaderGraph, GraphLoaderConfig

mock_config = {
    "n": 10,               # 10 stations
    "n_depots": 2,         # 2 depots
    "n_timestamps": 168,   # 1 week of hourly data
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "seed": 42,
}

mock = DataLoaderMock(mock_config)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
loader.load_data()

resolved = loader.resolved
print(f"Facilities: {len(resolved.facilities)}")
print(f"Periods:    {len(resolved.periods)}")
print(f"Edges:      {len(resolved.edges)}")
print(f"Resources:  {len(resolved.resources)}")
print(f"\nFacility types:")
display(resolved.facilities[["facility_id", "facility_type", "name"]].head(14))

## 1. Inspect initial state

In [ ]:
from gbp.consumers.simulator.state import init_state

state = init_state(resolved)
print(f"Period: {state.period_id} (index={state.period_index})")
print(f"Total initial inventory: {state.inventory['quantity'].sum():.0f} bikes")
print(f"\nInventory per facility:")
display(state.inventory)
print(f"\nResources ({len(state.resources)}):")
display(state.resources[["resource_id", "status", "current_facility_id", "capacity_override"]])

## 2. Run simulation — NoopTask (no rebalancing, demand-only)

In [ ]:
from gbp.consumers.simulator import Environment, EnvironmentConfig
from gbp.consumers.simulator.built_in_phases import DemandPhase, ArrivalsPhase
from gbp.consumers.simulator.dispatch_phase import DispatchPhase
from gbp.consumers.simulator.tasks.noop import NoopTask

config = EnvironmentConfig(
    phases=[
        DemandPhase(),
        ArrivalsPhase(),
        DispatchPhase(task=NoopTask()),
    ],
)
env = Environment(resolved, config)
log = env.run()
dfs = log.to_dataframes()

print("Simulation complete. Log tables:")
for name, df in dfs.items():
    print(f"  {name}: {df.shape}")

## 3. Inventory evolution (no demand — inventory stays flat)

In [ ]:
inv = dfs["simulation_inventory_log"]
pivot = inv.pivot_table(index="period_id", columns="facility_id", values="quantity")
print(f"Inventory shape: {pivot.shape} (periods x facilities)")
print(f"\nFirst 5 periods:")
display(pivot.head())
print(f"\nTotal bikes per period (should be constant without demand):")
print(inv.groupby("period_id")["quantity"].sum().to_string())

## 4. Inject synthetic demand from mock trips and re-run

`DataLoaderGraph` does not generate demand from mock data.
We can derive daily demand per station from the trip data and inject it into `ResolvedModelData`.

In [ ]:
import dataclasses
import pandas as pd

# Derive daily demand per station from mock trip departures
trips = mock.df_trips.copy()
trips["date"] = trips["started_at"].dt.normalize()

demand_agg = (
    trips
    .groupby(["start_station_id", "date"])
    .size()
    .reset_index(name="quantity")
    .rename(columns={"start_station_id": "facility_id"})
)
demand_agg["commodity_category"] = "working_bike"
demand_agg["quantity_unit"] = "bike"
demand_agg["quantity"] = demand_agg["quantity"].astype(float)

print(f"Generated {len(demand_agg)} demand rows from {len(trips)} trips")
print(f"Date range: {demand_agg['date'].min()} .. {demand_agg['date'].max()}")
display(demand_agg.head(10))

In [ ]:
# Rebuild resolved model with demand injected into raw
raw_with_demand = dataclasses.replace(loader.raw, demand=demand_agg)
from gbp.build.pipeline import build_model

resolved_wd = build_model(raw_with_demand)

print(f"Demand rows (resolved): {len(resolved_wd.demand)}")
print(f"Demand per period:")
display(resolved_wd.demand.groupby("period_id")["quantity"].sum())

## 5. Re-run simulation with demand — inventory should decrease

In [ ]:
env_wd = Environment(resolved_wd, config)
log_wd = env_wd.run()
dfs_wd = log_wd.to_dataframes()

print("Simulation with demand complete. Log tables:")
for name, df in dfs_wd.items():
    print(f"  {name}: {df.shape}")

In [ ]:
# Inventory evolution with demand — total bikes should decrease
inv_wd = dfs_wd["simulation_inventory_log"]
pivot_wd = inv_wd.pivot_table(index="period_id", columns="facility_id", values="quantity")

print("Inventory per period (stations drain, depot stays):")
display(pivot_wd)

print(f"\nTotal bikes per period:")
totals = inv_wd.groupby("period_id")["quantity"].sum()
print(totals.to_string())

In [ ]:
# Flow events — demand consumption
flow = dfs_wd["simulation_flow_log"]
print(f"Flow events: {len(flow)}")
if not flow.empty:
    display(flow.head(10))

In [ ]:
# Unmet demand — stations that ran out of bikes
unmet = dfs_wd["simulation_unmet_demand_log"]
if unmet.empty:
    print("No unmet demand — all stations had enough bikes.")
else:
    print(f"Unmet demand: {len(unmet)} events, total deficit: {unmet['deficit'].sum():.0f} bikes")
    display(unmet)

## 6. Compare mock source data vs simulation

The mock source has hourly trip-level data. The simulation operates on daily periods.
Let's compare the original mock inventory trajectory with the simulation result.

In [ ]:
# Mock source: hourly inventory (first timestamp per day vs simulation)
mock_inv = mock.df_inventory_ts.copy()
mock_inv.index.name = "timestamp"

# Daily snapshot: take first hour of each day
mock_daily = mock_inv.resample("D").first()
print(f"Mock daily inventory (first hour per day), shape: {mock_daily.shape}")
display(mock_daily)

print(f"\nSimulation inventory (per period):")
display(pivot_wd)

## 7. Summary statistics

In [ ]:
print("=== Mock Data Summary ===")
print(f"  Stations: {len(mock.df_stations)}")
print(f"  Depots:   {len(mock.df_depots)}")
print(f"  Trucks:   {len(mock.df_resources)}")
print(f"  Trips:    {len(mock.df_trips)}")
print(f"  Hours:    {len(mock.timestamps)}")

print(f"\n=== Resolved Model ===")
print(f"  Facilities: {len(resolved_wd.facilities)}")
print(f"  Edges:      {len(resolved_wd.edges)}")
print(f"  Periods:    {len(resolved_wd.periods)}")
print(f"  Resources:  {len(resolved_wd.resources)}")
print(f"  Demand rows: {len(resolved_wd.demand)}")

print(f"\n=== Simulation Results ===")
total_demand = resolved_wd.demand["quantity"].sum()
total_fulfilled = flow["quantity"].sum() if not flow.empty else 0
total_unmet = unmet["deficit"].sum() if not unmet.empty else 0
print(f"  Total demand:    {total_demand:.0f} bikes")
print(f"  Total fulfilled: {total_fulfilled:.0f} bikes")
print(f"  Total unmet:     {total_unmet:.0f} bikes")
print(f"  Fill rate:       {total_fulfilled / total_demand * 100:.1f}%" if total_demand > 0 else "  No demand")